# Qwen2.5-Omni — Clotho audio-captioning smoke test

Self-contained 5-clip smoke test to validate the LALM path on **your** Colab GPU
before the full 1045-clip run. It auto-detects the GPU and picks the 3B/7B model
size + precision that fits, captions 5 Clotho clips, and writes the project's
predictions-JSON schema.

**Before running:**
1. Upload the Clotho eval audio folder + `clotho_captions_evaluation.csv` to your
   Google Drive (one-time, ~2 GB).
2. Set `DRIVE_CLOTHO` in the *Paths* cell to where you put it.
3. Run top to bottom. **Restart the runtime after the install cell** (Colab pins an
   old transformers; the upgrade needs a restart).


## 1 · Install (then RESTART the runtime)

In [ ]:
# Qwen2.5-Omni needs a recent transformers + the omni helper. Try mainline first.
!pip install -q -U "transformers>=4.52" accelerate "qwen-omni-utils[decord]" soundfile librosa
print("\nInstalled. NOW do: Runtime > Restart session, then run the cells below.")
print("If a later cell raises KeyError: 'qwen2_5_omni', install the preview branch instead:")
print('  !pip install -q -U "git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview" "qwen-omni-utils[decord]"')


## 2 · Detect GPU and auto-pick model size/precision

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU. In Colab: Runtime > Change runtime type > GPU.")

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} | VRAM: {vram:.1f} GB")

# Audio-only, bf16, talker disabled. 7B needs ~24 GB+; otherwise use 3B.
MODEL_ID = "Qwen/Qwen2.5-Omni-7B" if vram >= 38 else "Qwen/Qwen2.5-Omni-3B"
CAPTION_PROMPT = "Describe the audio in one sentence."
print("Selected model:", MODEL_ID)
print("Prompt        :", CAPTION_PROMPT)


## 3 · Paths (EDIT `DRIVE_CLOTHO`) + load 5 clips

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pathlib, pandas as pd

# <-- EDIT THIS to wherever you uploaded the Clotho eval data on your Drive:
DRIVE_CLOTHO = "/content/drive/MyDrive/clotho_v2.1"

root = pathlib.Path(DRIVE_CLOTHO)
AUDIO_DIR = root / "clotho_audio_evaluation" / "evaluation"
df = pd.read_csv(root / "clotho_captions_evaluation.csv")
assert AUDIO_DIR.is_dir(), f"audio dir not found: {AUDIO_DIR}"
sample = df.head(5)
print("clips:", sample["file_name"].tolist())


## 4 · Load the model (text-only, talker disabled to save VRAM)

In [ ]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
model.disable_talker()        # text-only output -> frees ~2 GB
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba "
          "Group, capable of perceiving auditory and visual inputs, as well as "
          "generating text and speech.")

@torch.inference_mode()
def caption(audio_path) -> str:
    conv = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": str(audio_path)},
            {"type": "text", "text": CAPTION_PROMPT},
        ]},
    ]
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True).to(model.device).to(model.dtype)
    out = model.generate(**inputs, return_audio=False, do_sample=False, max_new_tokens=64)
    gen = out[:, inputs["input_ids"].shape[1]:]   # keep only newly generated tokens
    txt = processor.batch_decode(gen, skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)[0]
    return txt.strip()

print("model loaded.")


## 5 · Smoke: caption 5 clips, write predictions JSON to Drive

In [ ]:
import json, time

items = []
for _, row in sample.iterrows():
    fn = row["file_name"]
    refs = [str(row[f"caption_{i}"]) for i in range(1, 6)]
    t0 = time.time()
    pred = caption(AUDIO_DIR / fn)
    print(f"{fn[:30]:32s} {time.time()-t0:5.1f}s  ->  {pred}")
    items.append({"file_name": fn, "prediction": pred, "references": refs})

payload = {"model": "qwen2_5_omni", "split": "evaluation",
           "decode": {"strategy": "greedy", "max_new_tokens": 64, "prompt": CAPTION_PROMPT},
           "items": items}
out_path = "/content/drive/MyDrive/qwen_smoke.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("\nsaved ->", out_path)
